<!-- # Retrieval Augmented Generation (RAG) with Azure AI Search and OpenAI

This code demonstrates how to work with RAG to give more context to the LLM/SLM models to get a more accurate answer. The code uses Azure AI Search to index the documents and Azure OpenAI's embedding model to generate embeddings/vectors for the documents.

+ Create an index schema
+ Load the sample data from a local folder
+ Embed the documents in-memory using Azure OpenAI's text-embedding-ada-002 model
+ Index the vector and non-vector fields on Azure AI Search
+ Run a series of vector and hybrid queries, including metadata filtering and hybrid (text + vectors) search. 

The code uses Azure OpenAI to generate embeddings for title and content fields. You'll need access to Azure OpenAI to run this demo.

## Create the resources

Refer to the `README.md` file in the root folder to create the resources. -->

<!-- ## Install python packages -->

# Retrieval Augmented Generation (RAG) with Azure AI Search and OpenAI

This code demonstrates how to work with RAG to give more context to the LLM/SLM models to get a more accurate answer. The code uses Azure AI Search to index the documents and Azure OpenAI's embedding model to generate embeddings/vectors for the documents.

+ Create an index schema
+ Load the sample data from a local folder
+ Embed the documents in-memory using Azure OpenAI's text-embedding-ada-002 model
+ Index the vector and non-vector fields on Azure AI Search
+ Run a series of vector and hybrid queries, including metadata filtering and hybrid (text + vectors) search. 

The code uses Azure OpenAI to generate embeddings for title and content fields. You'll need access to Azure OpenAI to run this demo.

## Create the resources

Refer to the `README.md` file in the root folder to create the resources.

## Install python packages

In [43]:
%pip install python-dotenv
%pip install tiktoken
%pip install azure-search-documents
%pip install azure-identity
%pip install openai


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


<!-- ## Connect to the Azure AI Search and OpenAI

Load environment variables from the `.env` file -->

## Connect to the Azure AI Search and OpenAI

Load environment variables from the `.env` file

In [44]:
import os
import re
from openai import AzureOpenAI
from dotenv import load_dotenv
from dotenv import dotenv_values

if os.path.exists(".env"):
    load_dotenv(override=True)
    config = dotenv_values(".env")

azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_chat_completions_deployment_name = os.getenv("AZURE_OPENAI_CHAT_COMPLETIONS_DEPLOYMENT_NAME")

azure_openai_embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
embedding_vector_dimensions = os.getenv("EMBEDDING_VECTOR_DIMENSIONS")

azure_search_service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
azure_search_service_admin_key = os.getenv("AZURE_SEARCH_SERVICE_ADMIN_KEY")
search_index_name = os.getenv("SEARCH_INDEX_NAME")

openai_client = AzureOpenAI(
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_api_key,
    api_version="2024-06-01"
)

# Test connection to OpenAI ChatGPT
completion = openai_client.chat.completions.create(
    model=azure_openai_chat_completions_deployment_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Who are you ?"}
    ])
print(completion.to_json())

NotFoundError: Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

<!-- ## Count the number of tokens in a text

Like LLM models, Embedding models defines a `max input`. It is defined in number of `tokens`. The `max_input` for `text-embedding-3-large` is 8191 tokens. So we need to split the text into chunks of 8191 tokens or less. For that, you need to get the number of tokens in a text string. -->

### Count the number of tokens in a text

Like LLM models, Embedding models defines a `max input`. It is defined in number of `tokens`. The `max_input` for `text-embedding-3-large` is 8191 tokens. So we need to split the text into chunks of 8191 tokens or less. For that, you need to get the number of tokens in a text string.

In [ ]:
import tiktoken

def num_tokens_from_string(string: str) -> int:
    encoding = tiktoken.get_encoding(encoding_name="cl100k_base")
    num_tokens = len(encoding.encode(string, disallowed_special=()))
    return num_tokens

# Test the function
num_tokens_from_string("tiktoken is great!")

6

<!-- The OpenAI embedding model `text-embedding-3-large` has a limit of `8191` tokens per request.
Before sending the files to the model, we need to split the text into chunks of less than `8191` tokens.
Count the number of tokens in the sample files and show the files with more than `8191` tokens. -->

In [ ]:
#!pip install PyPDF2

In [ ]:
import os
import PyPDF2
import pandas as pd

input_directory = './data/azure-ai-docs/'
i = 0

def num_tokens_from_string(string):
    # Assuming this function calculates tokens (you can define it as needed)
    return len(string.split())

for filename in os.listdir(input_directory):
    file_path = os.path.join(input_directory, filename)
    
    if filename.endswith('.pdf'):
        with open(file_path, 'rb') as file:  # Open in binary mode
            reader = PyPDF2.PdfReader(file)
            content = ''
            for page in reader.pages:
                content += page.extract_text() or ''  # Extract text from each page
            tokens = num_tokens_from_string(content)
            if tokens > 8191:
                print(f'File {filename} (PDF) has {tokens} tokens which is more than 8191 (max) tokens')
    
    elif filename.endswith('.xlsx') or filename.endswith('.xls'):  # Check for Excel files
        try:
            df = pd.read_excel(file_path, sheet_name=None)  # Read all sheets
            content = ''
            for sheet_name, sheet_data in df.items():
                content += '\n'.join(sheet_data.astype(str).values.flatten())  # Convert all data to string and flatten
            tokens = num_tokens_from_string(content)
            if tokens > 8191:
                print(f'File {filename} (Excel) has {tokens} tokens which is more than 8191 (max) tokens')
        except Exception as e:
            print(f"Error reading {filename}: {e}")


File 3pm Indicator Guide.xlsx (Excel) has 16689 tokens which is more than 8191 (max) tokens


c:\anaconda\Lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


<!-- ## Transforming/cleaning the documents

Later in this lab, we will proceed with markdown `.md` files. We will need to remove all special characters and markdown syntax from the files. The function `clean_markdown_content()` will help us with this. -->

## Transforming/cleaning the documents

Later in this lab, we will proceed with pdf and excel files. We will need to remove all special characters and markdown syntax from the files.

In [ ]:
import os
import re
import PyPDF2
import pandas as pd

# Function to count tokens (assuming it's word count for simplicity)
def num_tokens_from_string(string):
    return len(string.split())

# Function to clean extracted content (generic for PDF and Excel)
def clean_content(content):
    # Remove links
    link_pattern = r'\[([^\[]+)\]\(([^\)]+)\)'
    content = re.sub(link_pattern, r'\1', content)

    # Remove images (Markdown image links)
    image_pattern = r'\!\[([^\[]*)\]\(([^\)]+)\)'
    content = re.sub(image_pattern, '', content)

    # Remove [doc][doc] references
    doc_reference_pattern = r'\[doc\]\[doc\]'
    content = re.sub(doc_reference_pattern, '', content)

    # Remove all occurrences of ** and newlines
    content = content.replace('**', '')
    content = content.replace('\n', ' ').strip()

    return content

# Path to the directory with PDFs and Excel files
input_directory = './data/azure-ai-docs/'

# Loop through all files in the directory
for filename in os.listdir(input_directory):
    file_path = os.path.join(input_directory, filename)
    
    if filename.endswith('.pdf'):
        # Process PDF files
        with open(file_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            content = ''
            for page in reader.pages:
                extracted_text = page.extract_text() or ''
                cleaned_text = clean_content(extracted_text)
                content += cleaned_text + ' '  # Add a space between pages

            tokens = num_tokens_from_string(content)
            if tokens > 8191:
                print(f'File {filename} (PDF) has {tokens} tokens, exceeding the limit of 8191.')

    elif filename.endswith('.xlsx') or filename.endswith('.xls'):  # Handle Excel files
        try:
            # Read Excel file (all sheets)
            df = pd.read_excel(file_path, sheet_name=None)  # Read all sheets
            content = ''
            for sheet_name, sheet_data in df.items():
                # Convert all data to string and flatten
                sheet_content = ' '.join(sheet_data.astype(str).values.flatten())
                cleaned_text = clean_content(sheet_content)
                content += cleaned_text + ' '  # Add a space between sheets

            tokens = num_tokens_from_string(content)
            if tokens > 8191:
                print(f'File {filename} (Excel) has {tokens} tokens, exceeding the limit of 8191.')
        except Exception as e:
            print(f"Error reading {filename}: {e}")

File 3pm Indicator Guide.xlsx (Excel) has 16696 tokens, exceeding the limit of 8191.


c:\anaconda\Lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


<!-- ## Get the vector embedding for an input text -->

## Get the vector embedding for an input text

In [ ]:
def get_embeddings_vector(text):

    response = openai_client.embeddings.create(
        input=text,
        model=azure_openai_embedding_model,
    )

    embedding = response.data[0].embedding

    return embedding

# Test the function
vector = get_embeddings_vector("Sample text")
print(vector)

[-0.012435130774974823, -0.04316585138440132, -0.009822873398661613, 0.011554595082998276, 0.006599131505936384, -0.013384154066443443, -0.04163958877325058, 0.059954747557640076, -0.019371801987290382, 0.0006316626095212996, 0.028959864750504494, 0.007949287071824074, 0.008849390782415867, -0.05157986655831337, 0.013932042755186558, 0.013256965205073357, -0.010253357701003551, 0.00492366636171937, 0.008017773739993572, -0.02305048704147339, -0.002491184277459979, 0.004666842985898256, -0.026592200621962547, 0.051892947405576706, 0.007430749014019966, -0.006525753531605005, -0.01613338477909565, 0.012797129340469837, 0.007919936440885067, 0.024635452777147293, 0.008986363187432289, 0.03978068009018898, -0.005650108680129051, -0.028294570744037628, 0.01490063313394785, 0.013755936175584793, 0.03093617968261242, 0.020193634554743767, 0.027844518423080444, 0.007939503528177738, 0.026044311001896858, 0.015389819629490376, -0.04449643939733505, -0.015487657859921455, -0.02007623016834259, 0

## Create file chunks

<!-- ## Create file chunks

This is where we split the markdown files in folder `./data/azure-ai-docs` into chunks. -->

In [ ]:
import uuid
import re
import json
import os
import fitz  # PyMuPDF for PDF reading
import pandas as pd
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

input_directory = './data/azure-ai-docs/'
output_directory = './data/chunks/'

# Create output directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

chunk_index = 0

def clean_content(text):
    """Remove unnecessary characters and clean text."""
    text = re.sub(r'\[doc\]\[doc\]', '', text)  # Remove redundant doc references
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespace
    return text.strip()

def split_into_chunks(text, max_tokens=100):
    """Split text into meaningful chunks based on sentence boundaries."""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    current_length = 0

    for sentence in sentences:
        sentence_tokens = len(sentence.split())
        if current_length + sentence_tokens <= max_tokens:
            current_chunk += " " + sentence
            current_length += sentence_tokens
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
            current_length = sentence_tokens

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

def extract_text_from_pdf(file_path):
    """Extract text from a PDF file."""
    with fitz.open(file_path) as doc:
        content = ""
        for page in doc:
            content += page.get_text()  # Extract text from each page
    return content

def extract_text_from_excel(file_path):
    """Extract text from an Excel file."""
    df = pd.read_excel(file_path, sheet_name=None)  # Read all sheets into a dictionary
    content = ""
    for sheet_name, sheet_data in df.items():
        content += ' '.join(sheet_data.astype(str).values.flatten()) + ' '
    return content

# Loop through each file in the directory
for filename in os.listdir(input_directory):
    file_path = os.path.join(input_directory, filename)

    if filename.endswith('.pdf'):
        # Process PDF files
        print(f"Processing {filename} (PDF)")
        content = extract_text_from_pdf(file_path)

    elif filename.endswith('.xlsx') or filename.endswith('.xls'):
        # Process Excel files
        print(f"Processing {filename} (Excel)")
        content = extract_text_from_excel(file_path)

    else:
        # Skip unsupported file types
        continue

    # Preview extracted content for debugging
    print("Extracted Content Preview:")
    print(content[:100])  # Print the first 100 characters for better preview

    # Check if there is any substantial content
    if len(content.strip()) == 0:
        print(f'No extractable text found in {filename}')
        continue

    # Clean and normalize content
    content = clean_content(content)

    # Split content into optimized chunks
    chunks = split_into_chunks(content)
    print(f"Total Chunks Generated: {len(chunks)}")

    # Process and save each chunk
    for chunk in chunks:
        chunk_content = chunk.strip()
        if len(chunk_content) == 0:
            continue  # Skip empty chunks

        chunk_index += 1

        # Embedding generation (you would need to implement the get_embeddings_vector function)
        vector = get_embeddings_vector(chunk_content)

        chunk_data = {
            "id": str(uuid.uuid4()),
            'filename': filename,
            'chunk_index': chunk_index,
            'chunk_content': chunk_content,
            'vector': vector
        }

        chunk_file_name = f'chunk_{chunk_index}_{filename}.json'.replace('?', '').replace(':', '').replace("'", '').replace('|', '').replace('/', '').replace('\\', '')

        # Write chunk into JSON file
        with open(f'{output_directory}/{chunk_file_name}', 'w') as f:
            json.dump(chunk_data, f)

print("Chunk extraction complete!")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\georg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Processing 3pm Indicator Guide.xlsx (Excel)
Extracted Content Preview:
Dataset Section Sub Section Data Element Definition Disaggregations HTS Optimization ANC Department 
Total Chunks Generated: 174
Processing DREAMS Dashboard Indicator Guide.pdf (PDF)
Extracted Content Preview:
 
 
DREAMS DASHBOARDS INDICATOR GUIDE 
 
 
Contents 
Overview of the DREAMS Dashboards .............
Total Chunks Generated: 31
Processing Indicator Frequency Table MER v2.7.pdf (PDF)
Extracted Content Preview:
Indicator Frequency & Type
Quarterly
Report 3 months of results for these indicators, as instructed 
Total Chunks Generated: 6
Processing Kenya Layering and Completion Table Final 22Nov2021_Kenya June 2022_Final.xlsx (Excel)


c:\anaconda\Lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


Extracted Content Preview:
nan COP21 DREAMS Layering and Intervention Completion Tables nan nan nan nan nan nan nan nan nan DRE
Total Chunks Generated: 33
Processing MER 2.7 Infographic.pdf (PDF)
Extracted Content Preview:
Viral Suppression
Health Systems
22. CXCA_TX
23. PMTCT_ART
24. TB_ART
25. TX_CURR
26. TX_ML
27. TX_N
Total Chunks Generated: 1
Chunk extraction complete!


In [ ]:
import uuid
import re
import json
import os
import fitz  # PyMuPDF for PDF reading
import pandas as pd
import tiktoken  # For precise token counting

input_directory = './data/azure-ai-docs/'
output_directory = './data/chunks/'

# Create output directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

chunk_index = 0

# Load the tokenizer for your model (for example, OpenAI's GPT-3/4 tokenizer)
tokenizer = tiktoken.get_encoding("cl100k_base")  # Adjust this for the appropriate model if needed

# Function to count tokens in a text
def count_tokens(text):
    return len(tokenizer.encode(text))

def extract_text_from_pdf(file_path):
    """Extract text from a PDF file."""
    with fitz.open(file_path) as doc:
        content = ""
        for page in doc:
            content += page.get_text()  # Extract text from each page
    return content

def extract_text_from_excel(file_path):
    """Extract text from an Excel file."""
    df = pd.read_excel(file_path, sheet_name=None)  # Read all sheets into a dictionary
    content = ""
    for sheet_name, sheet_data in df.items():
        content += ' '.join(sheet_data.astype(str).values.flatten()) + ' '
    return content

def split_into_chunks_by_tokens(text, max_tokens=8191):
    """Split text into chunks where each chunk has fewer tokens than the max_tokens."""
    sentences = re.split(r'(?<=\.)\s', text)  # Split by sentences (simple heuristic)
    chunks = []
    current_chunk = ""
    current_token_count = 0

    for sentence in sentences:
        sentence_token_count = count_tokens(sentence)
        if current_token_count + sentence_token_count <= max_tokens:
            current_chunk += " " + sentence
            current_token_count += sentence_token_count
        else:
            # If adding this sentence would exceed the limit, save the current chunk and start a new one
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence
            current_token_count = sentence_token_count

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

# Loop through each file in the directory
for filename in os.listdir(input_directory):
    file_path = os.path.join(input_directory, filename)

    if filename.endswith('.pdf'):
        # Process PDF files
        print(f"Processing {filename} (PDF)")
        content = extract_text_from_pdf(file_path)

    elif filename.endswith('.xlsx') or filename.endswith('.xls'):
        # Process Excel files
        print(f"Processing {filename} (Excel)")
        content = extract_text_from_excel(file_path)

    else:
        # Skip unsupported file types
        continue

    # Preview extracted content for debugging
    print("Extracted Content Preview:")
    print(content[:10])  # Print the first 10 characters

    # Check if there is any substantial content
    if len(content.strip()) == 0:
        print(f'No extractable text found in {filename}')
        continue

    # Normalize content to handle inconsistent formatting
    content = content.replace('\r', ' ').replace('\n\n', '\n')

    # Split content into chunks by token count
    chunks = split_into_chunks_by_tokens(content)
    print(f"Total Chunks Generated: {len(chunks)}")

    # Process and save each chunk
    for chunk in chunks:
        chunk_content = chunk.strip()
        if len(chunk_content) == 0:
            continue  # Skip empty chunks

        chunk_index += 1

        # Embedding generation (You need to implement `get_embeddings_vector` function)
        vector = get_embeddings_vector(chunk_content)

        chunk_data = {
            "id": str(uuid.uuid4()),
            'filename': filename,
            'chunk_index': chunk_index,
            'chunk_content': chunk_content,
            'vector': vector
        }

        chunk_file_name = f'chunk_{chunk_index}_{filename}.json'.replace('?', '').replace(':', '').replace("'", '').replace('|', '').replace('/', '').replace('\\', '')

        # Write chunk into JSON file
        with open(f'{output_directory}/{chunk_file_name}', 'w') as f:
            json.dump(chunk_data, f)

print("Chunk extraction complete!")


Processing 3pm Indicator Guide.xlsx (Excel)
Extracted Content Preview:
Dataset Se
Total Chunks Generated: 3
Processing DREAMS Dashboard Indicator Guide.pdf (PDF)
Extracted Content Preview:
 
 
DREAMS
Total Chunks Generated: 1
Processing Indicator Frequency Table MER v2.7.pdf (PDF)
Extracted Content Preview:
Indicator 
Total Chunks Generated: 1
Processing Kenya Layering and Completion Table Final 22Nov2021_Kenya June 2022_Final.xlsx (Excel)


c:\anaconda\Lib\site-packages\openpyxl\worksheet\_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


Extracted Content Preview:
nan COP21 
Total Chunks Generated: 1
Processing MER 2.7 Infographic.pdf (PDF)
Extracted Content Preview:
Viral Supp
Total Chunks Generated: 1
Chunk extraction complete!


### Create Index in Azure AI Search.

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    ComplexField,
    CorsOptions,
    SearchIndex,
    SearchField,
    ScoringProfile,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticSearch,
    SemanticField,
    LexicalAnalyzerName
)

# Set credentials and initialize client
credential = AzureKeyCredential(azure_search_service_admin_key)

search_index_client = SearchIndexClient(
    endpoint=azure_search_service_endpoint, 
    index_name=search_index_name, 
    credential=credential
)

# 🔥 Delete existing index if it already exists
try:
    existing_index = search_index_client.get_index(search_index_name)
    if existing_index:
        search_index_client.delete_index(search_index_name)
        print(f"Deleted existing index: {search_index_name}")
except Exception as e:
    print(f"No existing index found or error occurred: {e}")

# ✅ Define fields for the new index with lowercase normalization
fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        sortable=True,
        filterable=True,
        facetable=True,
    ),
    SearchableField(name="page_title", type=SearchFieldDataType.String, analyzer_name=LexicalAnalyzerName.EN_LUCENE),
    SearchableField(name="page_description", type=SearchFieldDataType.String, analyzer_name=LexicalAnalyzerName.EN_LUCENE),
    SearchableField(name="page_date", type=SearchFieldDataType.String, analyzer_name=LexicalAnalyzerName.EN_LUCENE),
    SearchableField(name="chunk_title", type=SearchFieldDataType.String, analyzer_name=LexicalAnalyzerName.EN_LUCENE),
    SearchableField(name="chunk_content", type=SearchFieldDataType.String, analyzer_name=LexicalAnalyzerName.EN_LUCENE),
    SearchField(
        name="vector", 
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=3072,  # Set dimension
        vector_search_profile_name="myHnswProfile",
    ),
]

# ✅ Configure vector search
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="myHnsw"
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw"
        )
    ]
)

# ✅ Configure semantic search
semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="page_title"),
        content_fields=[SemanticField(field_name="chunk_content")]
    )
)

semantic_search = SemanticSearch(configurations=[semantic_config])

# ✅ Define and create the search index
search_index = SearchIndex(
    name=search_index_name, 
    fields=fields, 
    vector_search=vector_search, 
    semantic_search=semantic_search
)

# ✅ Create or update the index on Azure
result = search_index_client.create_or_update_index(search_index)
print(f"{result.name} created successfully.")

Deleted existing index: index-doc
index-doc created successfully.


<!-- ## Upload chunks/documents to Azure AI Search -->

### Upload chunks/documents to Azure AI Search

In [ ]:
import uuid
import os
import json
from azure.search.documents import SearchClient
from azure.core.exceptions import HttpResponseError

# Initialize the SearchClient
search_client = SearchClient(
    endpoint=azure_search_service_endpoint,
    index_name=search_index_name,
    credential=credential
)

# Define batch size to prevent 'Request Entity Too Large' error
BATCH_SIZE = 500  # Adjust if necessary based on document size

# Define allowed fields based on your index schema
allowed_fields = {
    "id", "page_title", "page_description", "page_date", "chunk_title", "chunk_content", "vector"
}

# Function to clean and filter documents based on allowed fields
def clean_document(doc):
    return {key: value for key, value in doc.items() if key in allowed_fields}

# Function to split documents into manageable batches
def batch_documents(documents, batch_size):
    for i in range(0, len(documents), batch_size):
        yield documents[i:i + batch_size]

# Prepare documents for upload
documents_batch = []

for filename in os.listdir(output_directory):
    if filename.endswith('.json'):
        with open(os.path.join(output_directory, filename), 'r') as file:
            document = json.load(file)
            cleaned_doc = clean_document(document)  # Clean document
            documents_batch.append(cleaned_doc)

# Upload documents in batches
try:
    if documents_batch:
        for batch_number, batch in enumerate(batch_documents(documents_batch, BATCH_SIZE), 1):
            result = search_client.upload_documents(documents=batch)
            for doc_result in result:
                print(f"Batch {batch_number} - Upload of document ID {doc_result.key} succeeded: {doc_result.succeeded}")
    else:
        print("No documents found for upload.")
except HttpResponseError as e:
    print(f"An error occurred during document upload: {e}")


Batch 1 - Upload of document ID 6625e405-b09e-423b-975c-db1bba968125 succeeded: True
Batch 1 - Upload of document ID 872f0ea9-f7e8-4c02-91f9-5a4b1330c857 succeeded: True
Batch 1 - Upload of document ID 96387f76-d561-42f0-ba6b-268114239221 succeeded: True
Batch 1 - Upload of document ID 9a3f3dbb-4222-48b3-8c7b-c377a8d40124 succeeded: True
Batch 1 - Upload of document ID 6dc23408-1c74-4ca7-8f48-e4a27b3fcfad succeeded: True
Batch 1 - Upload of document ID ca45eb7d-0cc3-4e5a-9ef1-8294f31bbbe8 succeeded: True
Batch 1 - Upload of document ID 90158030-b9ae-4f6b-89f0-766ce1ea72b2 succeeded: True
Batch 1 - Upload of document ID 70f7d694-6639-43c1-b781-191ae3b56aa2 succeeded: True
Batch 1 - Upload of document ID a6e15310-5ed8-424b-93d9-0d275ed0f8e0 succeeded: True
Batch 1 - Upload of document ID 48190cc7-21d8-4d53-8df3-4ffeb069e4e4 succeeded: True
Batch 1 - Upload of document ID a97c7458-b8ad-4526-969e-89f0a31da39a succeeded: True
Batch 1 - Upload of document ID 8203b139-5213-4052-ad2a-aeeb43d46

<!-- ## Perform a vector similarity search

This example shows a pure vector search using the vectorizable text query, all you need to do is pass in text and your vectorizer will handle the query vectorization. -->

In [ ]:
from azure.search.documents.models import VectorizedQuery

# Pure Vector Search
query = "Explain what is TX_CURR ?"  

embedding = get_embeddings_vector(query)

vector_query = VectorizedQuery(vector=embedding, k_nearest_neighbors=3, fields="vector")
  
results = search_client.search(  
    search_text=None,  
    vector_queries= [vector_query],
    select=[
            #page_title", "page_date", 
            "chunk_title", 
            "chunk_content"],
)  
  
for result in results:
    print(f"-------------------------------------------")
    #print(f"Page Date: {result['page_date']}")  
    #print(f"Page Title: {result['page_title']}")  
    print(f"Chunk Title: {result['chunk_title']}")  
    print(f"Chunk Content: {result['chunk_content']}")
    print(f"Score: {result['@search.score']}")  


-------------------------------------------
Chunk Title: None
Chunk Content: If it is determined that a client has died, they should also be immediately removed from the Current on treatment number. Age and Sex nan Care and Treatment Current on treatment - (Number with Diabetes) These are the number of PLHIV who have been diagnosed with Diabetes.
Score: 0.585258
-------------------------------------------
Chunk Title: None
Chunk Content: Age and Sex nan Care and Treatment Previously lost but returned to treatment this month (Tx_RTT) This is the number of ART patients who experienced Interruptions In Treatment (IIT) during any previous reporting period, who successfully restarted ARVs within the reporting period and remained on treatment until the end of the reporting period. Age and Sex nan Care and Treatment Transfer ins this month This is a sum of all clients who were initiated on HIV care and treatment services at another health facility and were transferred to this facility within 

<!-- ## Simulate a user query

This is where we will use the Azure AI Search to search for documents similar to the user query. -->

## Simulate a user query

This is where we will use the Azure AI Search to search for documents similar to the user query.

In [ ]:
response = openai_client.chat.completions.create(
    model=azure_openai_chat_completions_deployment_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant for an AI learner."},
        {"role": "user", "content": "Explian the meaning of TX_CURR Indicator?"}
    ],
    extra_body={
        "data_sources": [
            {
                "type": "azure_search",
                "parameters": {
                    "endpoint": azure_search_service_endpoint,
                    "index_name": search_index_name,
                    "authentication": {
                        "type": "api_key",
                        "key": azure_search_service_admin_key,
                    }
                }
            }
        ]
    }
)

print(response.to_json())

{
  "id": "922cc07c-2802-4be5-a6c9-5680040a0ffa",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "The TX_CURR indicator refers to the number of individuals currently receiving antiretroviral therapy (ART) at a specific point in time. This indicator is reported quarterly and is used to monitor the ongoing treatment of people living with HIV. It includes new and continuing patients but excludes those who have died, transferred out, or cannot be accounted for (i.e., have not been seen in the last 3 months) [doc4].",
        "role": "assistant",
        "end_turn": true,
        "context": {
          "citations": [
            {
              "content": "PEPFAR MER v2.7 Indicator Frequency Table QUARTERLY HTS_TST Ⓕ Ⓒ HTS_INDEX Ⓕ Ⓒ HTS_RECENT Ⓕ Ⓒ HTS_SELF Ⓕ Ⓒ PMTCT_ART Ⓕ PMTCT_EID Ⓕ PMTCT_HEI Ⓕ PMTCT_STAT Ⓕ PrEP_CT Ⓕ PrEP_NEW Ⓕ TB_STAT Ⓕ TX_CURR Ⓕ TX_ML Ⓕ TX_NEW Ⓕ TX_PVLS Ⓕ TX_RTT Ⓕ VMMC_CIRC Ⓕ SEMI-ANNUAL AGYW_PREV Ⓒ CXCA_SCRN Ⓕ

In [ ]:
print(response.choices[0].message.content)

The TX_CURR indicator refers to the number of individuals currently receiving antiretroviral therapy (ART) at a specific point in time. This indicator is reported quarterly and is used to monitor the ongoing treatment of people living with HIV. It includes new and continuing patients but excludes those who have died, transferred out, or cannot be accounted for (i.e., have not been seen in the last 3 months) [doc4].
